In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import warnings
import pickle
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import multiprocessing

In [3]:
sys.path.append('../')

In [4]:
import friendly.ellipses as ellipses
import friendly.entropy as entropy
import friendly.gaussians as gaussians
import friendly.graphs as graphs
import friendly.matching as matching

In [5]:
warnings.filterwarnings('ignore')

### 1. Load cosmoDC2 and DC2object with few cuts

In [6]:
tract = 4026

In [7]:
quantities_true = ['ra', 'dec', 'mag_i', 'size_true', 'size_minor_true', 'position_angle_true']

quantities_obj = ['ra', 'dec', 'blendedness', 'mag_i_cModel', 'x', 'y', 'Ixx_pixel_i', 'Iyy_pixel_i', 'Ixy_pixel_i', 'patch', 'psf_fwhm_i']

In [ ]:
filters_true = [(np.isfinite, 'mag_i'), 'mag_i>18', 'mag_i<29', (np.isfinite,'ellipticity_1_true'), (np.isfinite,'ellipticity_2_true'), 
                (np.isfinite,'redshift'), (np.isfinite, 'convergence'), (np.isfinite, 'mag_i'), (np.isfinite, 'shear_1'),
                (np.isfinite, 'shear_2')]

filters_obj = [(np.isfinite, 'mag_i_cModel'), 'mag_i_cModel > 18', 'mag_i_cModel < 26.5', (np.isfinite, 'mag_i_cModel'), 
               (np.isfinite,'ext_shapeHSM_HsmShapeRegauss_e1'), (np.isfinite,'ext_shapeHSM_HsmShapeRegauss_e2'), 
               (np.isfinite, 'ext_shapeHSM_HsmShapeRegauss_sigma'), ('ext_shapeHSM_HsmShapeRegauss_flag == 0'),
               (np.isfinite, 'photoz_mean'), (np.isfinite, 'blendedness')]

In [ ]:
cosmodc2 = GCRCatalogs.load_catalog('cosmoDC2_v1.1.4_image') #truth catalog
dc2 = GCRCatalogs.load_catalog('dc2_object_run2.2i_dr6_with_addons') #object catalog

In [ ]:
object_data = pd.DataFrame(dc2.get_quantities(quantities_obj,
                                  filters=['extendedness>0', 'clean']+filters_obj,
                                  native_filters=[f'tract=={tract}']))

In [ ]:

eps = 10/3600 # 10 arcsec
max_ra = np.nanmax(object_data['ra']) + eps
min_ra = np.nanmin(object_data['ra']) - eps
max_dec = np.nanmax(object_data['dec']) + eps
min_dec = np.nanmin(object_data['dec']) - eps
pos_filters = [f'ra >= {min_ra}',f'ra <= {max_ra}', f'dec >= {min_dec}', f'dec <= {max_dec}']

vertices = hp.ang2vec(np.array([min_ra, max_ra, max_ra, min_ra]),
                      np.array([min_dec, min_dec, max_dec, max_dec]), lonlat=True)
ipix = hp.query_polygon(32, vertices, inclusive=True)
healpix_filter = GCRQuery((lambda h: np.isin(h, ipix, True), "healpix_pixel"))

In [ ]:

truth_data = pd.DataFrame(cosmodc2.get_quantities(quantities_true, filters=filters_true+pos_filters, 
                                      native_filters=healpix_filter))

In [ ]:

print("number of galaxies =", len(truth_data['ra']))
print("number of objects =", len(object_data['ra']))

### 2. FoF groups

In [8]:
FoF_groups = matching.FoF_matching(truth_data, object_data, 2.) #takes time

### 3. Friendly graphs on one tract

In [10]:
def _map_f(args):
    f, i = args
    return f(i)

In [11]:
def map(func, iter, ordered=True, n=None):
    """
    Maps the function `func` over the iterator `iter` in a multi-threaded way
    using the multiprocessing package with a tqdm progress bar.


    Parameters
    ----------
    func : function
        func must be pickable, see https://docs.python.org/2/library/pickle.html#what-can-be-pickled-and-unpickled .
    iter : iterator
    ordered : bool
        Whether to use imap (default) or imap_unordered

    Returns
    -------
    type
        Result of the computation of func(iter).
    """

    pool = multiprocessing.Pool()
    print(multiprocessing.cpu_count())

    inputs = ((func,i) for i in iter) #use a generator, so that nothing is computed before it's needed :)

    try :
        n = len(iter)
    except TypeError : # if iter is a generator
        pass

    res_list = []

    if ordered:
        pool_map = pool.imap
    else:
        pool_map = pool.imap_unordered
    
    with tqdm(total=n, desc='# castor.parallel.map') as pbar:
        for res in pool_map(_map_f, inputs):
            try :
                pbar.update()
                res_list.append(res)
            except KeyboardInterrupt:
                pool.terminate()

    pool.close()
    pool.join()
    
    return res_list

In [12]:
def func(args):
    
    group, dtruth, dobj, link_type = args
    
    G = graphs.group2graph(group, dtruth, dobj, link_type)
    
    return G

In [13]:
np.random.shuffle(FoF_groups)

In [14]:
iter = ((group, truth_data.loc[group[0]], object_data.loc[group[1]]) for group in FoF_groups)

In [ ]:
glist = map(func, iter, n=len(FoF_groups))